# Validating the dataset

In [19]:
import json
import re
import requests
from urllib.parse import urlsplit

data_path = "../data"

with open(f"{data_path}/quizzes.json", "r") as f:
    quizzes = json.load(f)

In [10]:
quizzes[0]

{'_id': 'KVicfAwDCYWseVLEXcQ1wgoc',
 'date': '2019-10-26',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F10%2F26',
 'headlines': [{'text': '\xa0“Hitman hires hitman who hires hitman who hires hitman who hires hitman who tells police',
   'url': 'https://metro.co.uk/2019/10/23/hitman-hires-hitman-hires-hitman-hires-hitman-hires-hitman-tells-police-10971438/'},
  {'text': 'Man’s body brews its own beer after yeast take over his gut microbiome',
   'url': 'https://www.newscientist.com/article/2220432-mans-body-brews-its-own-beer-after-yeast-take-over-his-gut-microbiome/'},
  {'text': 'A parking space in Hong Kong just sold for almost $1 million',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-24/hong-kong-car-parking-spot-sells-for-eye-watering-1-million'},
  {'text': 'Sanders staffer fired after accidentally ordering Uber helicopter to Queens rally',
   'url': 'https://media.giphy.com/media/l0NhWLdqrEy4hgeGc/giphy.gif'}],
 'answer': 'We made up the Bernie news.'}

Automated structural checks

In [ ]:
date_search = re.compile(r'^\d{4}-\d{2}-\d{2}$')
dates_list = []
ids_list = []

for quiz in quizzes:
    print(f"Checking quiz {quiz["date"]}")
    if not quiz["_id"]:
        print(f"{quiz["date"]} edition does not have an id")
    else:
        ids_list.append(quiz["_id"])
    if not date_search.search(quiz["date"]):
        print(f"{quiz["date"]} edition has invalid date")
    else:
        dates_list.append(quiz["date"])
    if (len(quiz["headlines"])) != 4:
        print(f"{quiz["date"]} edition does not have four headlines")
    for headline in quiz["headlines"]:
        url = headline["url"]
        if not headline["text"]:
            print(f"{quiz["date"]} edition has empty headline")
        if not url:
            print(f"{quiz["date"]} edition has empty URL")
        else:
            parts = urlsplit(url)
            if not parts.scheme:
                print(f"{url} edition has no scheme")
            if not parts.netloc:
                print(f"{url} edition has no domain")
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 404:
                    print(f"{url} returned {response.status_code}")
            except requests.RequestException as e:
                print(f"{url} faild: {e}")
    if not quiz["answer"]:
        print(f"{quiz["date"]} edition has empty answer")

ids_set = set(ids_list)
if len(ids_set) == len(ids_list):
    print("No duplicate ids detected")
else:
    print("Duplicate ids detected")
dates_set = set(dates_list)
if len(dates_set) == len(dates_list):
    print("No duplicate dates detected")
else:
    print("Duplicate dates detected")



Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Checking quiz 2019-11-23
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-11
https://abcnews.go.com/Technology/weird-smart-home-gadgets-ces-2020-toilet-paper/story returned 404
Checking quiz 2020-01-18
Checking quiz 2020-01-25
Checking quiz 2020-02-01
http://www.asahi.com/ajw/articles/AJ202001260004.html returned 404
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Checking quiz 2020-02-22
Checking quiz 2020-02-29
Checking quiz 2020-03-07
https://www.washingtonpost.com/nation/2020/03/04/jill-biden-joe-biden-protesters/ faild: HTTPSConnectionPool(host='www.washingtonpost.com', port=443): Read timed out. (read timeout=10)
Checking quiz 2020-03-14
Checking quiz 2020-03-21
Checking quiz 2020-03-28
Checking quiz 2020-04-04
Checking quiz 2020-04-18
Checking quiz 2020-04-25
Checking quiz 2020-05-02
Checking quiz 2020-05-0

Automated anomaly detection

In [21]:
quizzes[1]

{'_id': 'z9gwGZ5GPFVVhEwb6cJdPxos',
 'date': '2019-11-02',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F11%2F02',
 'headlines': [{'text': 'Leading maggot farmer to expand from Cape Town to California',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-31/leading-maggot-farmer-to-expand-from-cape-town-to-california'},
  {'text': 'Riding wave of support for foie gras ban, de Blasio re-enters presidential race',
   'url': 'https://media.giphy.com/media/PoLEEuKQT7FnnJRiAA/giphy.gif'},
  {'text': '“Migrating Russian eagles run up huge data roaming charges"',
   'url': 'https://www.bbc.com/news/world-europe-50180781'},
  {'text': 'Horsepower, literally: Finnish horse show runs on manure',
   'url': 'https://apnews.com/3ad8a6b9b1cb4100a9db1b2e406ad14d'}],
 'answer': 'Bill de Blasio is not re-entering the presidential race.'}

In [20]:
anomalous_characters = ["[", "]", "\n", "\xa0"]

for quiz in quizzes:
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        if len(text) > 120:
            print(f"Long headline text detected in {quiz["date"]} edition: {text}")
        if len(text) < 40:
            print(f"Short headline text detected in {quiz["date"]} edition: {text}")
        if any(character in text for character in anomalous_characters):
            print(f"Anomalous characters detected in {quiz["date"]} edition: {text}")
        if text.startswith(" "):
            print(f"{quiz["date"]} edition starts with whitespace: {text}")
        if text.endswith(" "):
            print(f"{quiz["date"]} edition ends with whitespace: {text}")
        if text.startswith("\"") and text.endswith("\""):
            print(f"{quiz["date"]} edition starts and ends with vertical quotation marks: {text}")

Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Short headline text detected in 2019-11-16 edition: Twinkies cereal is now in the works
Checking quiz 2019-11-23
Short headline text detected in 2019-11-23 edition: Why the world is running out of sand
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Short headline text detected in 2019-12-14 edition: ‘My boss lets us book hangover days’
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-11
Short headline text detected in 2020-01-11 edition: A toilet paper delivery bot
Short headline text detected in 2020-01-11 edition: A karaoke kit for your car
Checking quiz 2020-01-18
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Short headline text detected in 2020-02-15 edition: Psychedelics on the stock exchange?
Checking quiz 2020-02-22
Checking quiz 2020-02-29
Checking quiz 2020-03-07
Checking quiz 2020-03-14
Checking qui

In [ ]:
anomalous_characters = ["[", "]", "\n", "\xa0"]

for quiz in quizzes:
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        if any(character in text for character in anomalous_characters):
            print(f"Anomalous characters detected in {quiz["date"]} edition: {text}")

In [7]:
len(quizzes[0]["headlines"][0]["text"])

89